# 第六章：多 Agent 协作

## 学习目标
- 理解多 Agent 架构的设计思路与适用场景
- 用子图（Subgraph）封装独立 Agent
- 实现 Supervisor 模式（主管调度多个 Agent）
- 用 `Command(goto=...)` 控制 Agent 间的动态路由
- 用 `Send` 实现并行任务分发

## 0. 环境准备

In [ ]:
from importlib.metadata import version
print(f"langgraph 版本: {version('langgraph')}")

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="qwen-plus",
    openai_api_base=os.environ["Qwen_API_BASE"],
    openai_api_key=os.environ["Qwen_API_KEY"],
)

## 1. 为什么需要多 Agent？

第四章我们学了单个 ReAct Agent：一个模型 + 一组工具，循环调用直到完成。这在简单场景下足够，但会遇到瓶颈：

| 问题 | 表现 |
|------|------|
| 工具太多 | 模型选择困难，容易调错工具 |
| 上下文混乱 | 所有对话历史堆在一起，不同任务的信息互相干扰 |
| 难以维护 | 一个 Agent 承担多种职责，修改一处影响全局 |

解决思路：**分而治之**。把不同职责交给不同 Agent，每个 Agent 只关注自己擅长的事。

### 两种主要模式

**Supervisor（主管模式）**：一个 Supervisor 节点统一调度，决定把任务交给哪个 Agent。Agent 完成后把结果交回 Supervisor，由 Supervisor 决定下一步。

```
用户 → Supervisor → Agent A → Supervisor → Agent B → Supervisor → 用户
```

**Peer-to-Peer（对等模式）**：Agent 之间直接传递控制权，没有中央调度。

```
用户 → Agent A → Agent B → Agent A → 用户
```

本章先学最实用的 Supervisor 模式，以及实现它的核心技术：子图、`Command`、`Send`。

## 2. 子图——Agent 的封装方式

在 LangGraph 中，每个 Agent 可以是一个独立的 `StateGraph`，编译后作为节点加入父图。这就是**子图（Subgraph）**。

好处：
- Agent 的内部逻辑对外透明，父图只看到输入和输出
- 可以独立开发、测试每个 Agent
- 父子图通过共享的 State key 自动传递数据

### 示例：研究员 + 写作者流水线

场景：用户给一个主题，"研究员" Agent 收集资料，"写作者" Agent 根据资料写摘要。

In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

In [ ]:
# --- 子图 1：研究员 Agent ---
# 接收用户问题，用 mock 搜索工具收集资料

def mock_search(query: str) -> str:
    """模拟搜索引擎返回结果"""
    data = {
        "量子计算": "量子计算利用量子力学原理进行计算。量子比特可以同时处于 0 和 1 的叠加态，使其在特定问题上指数级快于经典计算机。Google 于 2019 年宣布实现量子优越性。",
        "人工智能": "人工智能是模拟人类智能的技术。当前主流方向是深度学习，基于大规模神经网络。2022 年以来，大语言模型（LLM）成为 AI 领域最热门的方向。",
    }
    for key, value in data.items():
        if key in query:
            return value
    return f"关于'{query}'的搜索结果：这是一个重要的技术领域，正在快速发展中。"


def researcher_node(state: MessagesState) -> dict:
    """研究员：收集资料"""
    user_query = state["messages"][-1].content
    search_result = mock_search(user_query)
    return {
        "messages": [
            AIMessage(content=f"[研究员] 收集到以下资料：\n{search_result}")
        ]
    }


# 构建研究员子图
researcher_builder = StateGraph(MessagesState)
researcher_builder.add_node("research", researcher_node)
researcher_builder.add_edge(START, "research")
researcher_builder.add_edge("research", END)
researcher_graph = researcher_builder.compile()

print("研究员子图：")
from IPython.display import Image
Image(researcher_graph.get_graph().draw_mermaid_png())

In [ ]:
# --- 子图 2：写作者 Agent ---
# 接收前面的消息历史（包含研究资料），写摘要

def writer_node(state: MessagesState) -> dict:
    """写作者：根据资料写摘要"""
    response = llm.invoke([
        SystemMessage(content="你是一个专业写手。根据对话中提供的资料，写一段简洁的摘要（3-4句话）。"),
        *state["messages"]
    ])
    return {
        "messages": [AIMessage(content=f"[写作者] {response.content}")]
    }


# 构建写作者子图
writer_builder = StateGraph(MessagesState)
writer_builder.add_node("write", writer_node)
writer_builder.add_edge(START, "write")
writer_builder.add_edge("write", END)
writer_graph = writer_builder.compile()

print("写作者子图：")
from IPython.display import Image
Image(writer_graph.get_graph().draw_mermaid_png())

In [ ]:
# --- 父图：串联两个子图 ---

parent_builder = StateGraph(MessagesState)
parent_builder.add_node("researcher", researcher_graph)  # 编译后的子图作为节点
parent_builder.add_node("writer", writer_graph)
parent_builder.add_edge(START, "researcher")
parent_builder.add_edge("researcher", "writer")
parent_builder.add_edge("writer", END)

pipeline = parent_builder.compile()
from IPython.display import Image
Image(pipeline.get_graph().draw_mermaid_png())

In [ ]:
# 运行流水线
result = pipeline.invoke({
    "messages": [HumanMessage(content="量子计算")]
})

for msg in result["messages"]:
    role = msg.__class__.__name__
    print(f"[{role}] {msg.content[:150]}")
    print()

### 刚才发生了什么？

1. 父图把 `HumanMessage("量子计算")` 传给 `researcher` 子图
2. 研究员子图内部执行 mock 搜索，返回 `AIMessage` 添加到消息列表
3. 父图把更新后的消息列表（包含用户问题 + 搜索结果）传给 `writer` 子图
4. 写作者子图调用 LLM 生成摘要

关键点：**父子图通过 `messages` 这个共享 key 自动传递数据**。因为两个子图和父图都用了 `MessagesState`，它们共享 `messages` 字段。父图执行子图时，会把自己当前的 `messages` 传入子图，子图的输出也会合并回父图的 `messages`。

如果父图和子图的 State schema 不同，只有同名的 key 会自动映射。

## 3. Supervisor 模式

流水线是固定顺序（A → B），但很多场景需要**动态决策**：根据用户输入，决定找哪个 Agent 处理。

Supervisor 模式的核心：
1. Supervisor 节点接收请求，用 LLM 判断应该交给哪个 Agent
2. 对应 Agent 执行后，结果返回 Supervisor
3. Supervisor 决定是否需要继续（交给其他 Agent 或直接结束）

### 示例：客服系统

场景：一个客服系统有两个专业 Agent：
- **订单助手**：查询订单状态
- **技术支持**：排查技术问题

Supervisor 根据用户问题把请求路由到对应 Agent。

In [ ]:
from typing import Literal
from langgraph.types import Command
import json

In [ ]:
# --- Mock 工具函数 ---

def query_order(order_id: str) -> str:
    """模拟查询订单"""
    orders = {
        "ORD001": "已发货，预计明天到达",
        "ORD002": "已签收",
        "ORD003": "处理中，预计 3 天后发货",
    }
    return orders.get(order_id, f"未找到订单 {order_id}")


def check_system_status(component: str) -> str:
    """模拟检查系统状态"""
    statuses = {
        "网络": "正常，延迟 15ms",
        "数据库": "正常，负载 30%",
        "支付": "异常，正在维护中，预计 2 小时后恢复",
        "登录": "正常",
    }
    return statuses.get(component, f"未知组件：{component}")

In [ ]:
# --- 订单助手 Agent ---

def order_agent(state: MessagesState) -> dict:
    """处理订单相关查询"""
    response = llm.invoke([
        SystemMessage(content=(
            "你是订单助手。根据用户问题，从对话中提取订单号并回复订单状态。\n"
            "可用订单：ORD001、ORD002、ORD003。\n"
            "如果用户提到了订单号，直接回复状态。如果没有，请用户提供。"
        )),
        *state["messages"]
    ])
    
    # 尝试提取订单号并查询
    content = state["messages"][-1].content if state["messages"] else ""
    for oid in ["ORD001", "ORD002", "ORD003"]:
        if oid in content or oid.lower() in content.lower():
            status = query_order(oid)
            return {
                "messages": [AIMessage(content=f"[订单助手] 订单 {oid} 状态：{status}")]
            }
    
    return {
        "messages": [AIMessage(content=f"[订单助手] {response.content}")]
    }

In [ ]:
# --- 技术支持 Agent ---

def tech_agent(state: MessagesState) -> dict:
    """处理技术支持查询"""
    content = state["messages"][-1].content if state["messages"] else ""
    
    # 检查所有相关组件
    components_to_check = []
    keyword_map = {
        "网络": "网络", "连不上": "网络", "断网": "网络",
        "数据库": "数据库", "数据": "数据库",
        "支付": "支付", "付款": "支付", "充值": "支付",
        "登录": "登录", "登不上": "登录", "密码": "登录",
    }
    for keyword, component in keyword_map.items():
        if keyword in content and component not in components_to_check:
            components_to_check.append(component)
    
    if not components_to_check:
        components_to_check = ["网络", "数据库", "支付", "登录"]
    
    results = []
    for comp in components_to_check:
        status = check_system_status(comp)
        results.append(f"  - {comp}：{status}")
    
    report = "[技术支持] 系统检查结果：\n" + "\n".join(results)
    return {"messages": [AIMessage(content=report)]}

In [ ]:
# --- Supervisor 节点 ---
# 用 LLM 决定路由到哪个 Agent，或直接结束

def supervisor(state: MessagesState) -> Command[Literal["order_agent", "tech_agent", "__end__"]]:
    """主管：根据用户问题决定路由"""
    response = llm.invoke([
        SystemMessage(content=(
            "你是客服主管。根据用户的问题，决定交给哪个 Agent 处理。\n"
            "可选 Agent：\n"
            "- order_agent：处理订单查询（如查快递、订单状态）\n"
            "- tech_agent：处理技术问题（如系统故障、登录问题、支付异常）\n"
            "- FINISH：如果问题已经被回答了，或者是闲聊，直接结束\n\n"
            "只回复一个词：order_agent 或 tech_agent 或 FINISH，不要说其他内容。"
        )),
        *state["messages"]
    ])
    
    decision = response.content.strip().lower()
    
    if "order" in decision:
        goto = "order_agent"
    elif "tech" in decision:
        goto = "tech_agent"
    else:
        goto = "__end__"
    
    print(f"  [Supervisor 决策] → {goto}")
    return Command(goto=goto)

In [ ]:
# --- 构建 Supervisor 图 ---

builder = StateGraph(MessagesState)

builder.add_node("supervisor", supervisor)
builder.add_node("order_agent", order_agent)
builder.add_node("tech_agent", tech_agent)

builder.add_edge(START, "supervisor")
# Agent 完成后回到 Supervisor（循环）
builder.add_edge("order_agent", "supervisor")
builder.add_edge("tech_agent", "supervisor")
# supervisor 通过 Command(goto=...) 决定下一步，不需要 add_edge 到 END

customer_service = builder.compile()
from IPython.display import Image
Image(customer_service.get_graph().draw_mermaid_png())

In [ ]:
# 测试 1：订单查询
result = customer_service.invoke({
    "messages": [HumanMessage(content="我的订单 ORD001 到哪了？")]
})

print("\n--- 最终结果 ---")
for msg in result["messages"]:
    if isinstance(msg, AIMessage):
        print(msg.content)

In [ ]:
# 测试 2：技术问题
result = customer_service.invoke({
    "messages": [HumanMessage(content="支付页面打不开，付款一直失败")]
})

print("\n--- 最终结果 ---")
for msg in result["messages"]:
    if isinstance(msg, AIMessage):
        print(msg.content)

### 刚才发生了什么？

整个流程：

```
用户提问 → Supervisor（LLM 判断路由）
         → order_agent 或 tech_agent（执行任务）
         → Supervisor（判断是否结束）
         → __end__
```

Supervisor 的关键职责：
1. **首次路由**：用户说"订单 ORD001 到哪了" → LLM 判断这是订单问题 → `Command(goto="order_agent")`
2. **二次判断**：Agent 回复后，消息列表里已经有了回答 → LLM 判断问题已解决 → `Command(goto="__end__")`

注意 `Command` 的使用：Supervisor 返回 `Command(goto=...)` 替代了 `conditional_edges`。这样路由逻辑完全在节点函数内部，更灵活。

## 4. Command(goto=...) 路由详解

上一节用了 `Command` 的基本形式，这里深入讲解它的能力。

### Command 的参数

| 参数 | 类型 | 说明 |
|------|------|------|
| `goto` | `str` 或 `list[str]` | 下一个要执行的节点（可以是多个） |
| `update` | `dict` | 同时更新 State（和路由一起生效） |

`Command` 把「路由」和「状态更新」合并成一个原子操作，避免了分两步操作可能出现的不一致。

### 对比：conditional_edges vs Command

先回顾第三章学过的 `conditional_edges`：

In [ ]:
# --- 方式 1：conditional_edges（静态路由表） ---

def route_by_topic(state: MessagesState) -> str:
    """路由函数：返回节点名称"""
    content = state["messages"][-1].content
    if "订单" in content:
        return "order"
    return "general"


def handle_order(state: MessagesState) -> dict:
    return {"messages": [AIMessage(content="这是订单处理结果")]}


def handle_general(state: MessagesState) -> dict:
    return {"messages": [AIMessage(content="这是通用回复")]}


g1 = StateGraph(MessagesState)
g1.add_node("router_node", lambda state: state)  # 空节点，只用于路由
g1.add_node("order", handle_order)
g1.add_node("general", handle_general)
g1.add_edge(START, "router_node")
g1.add_conditional_edges("router_node", route_by_topic)  # 路由逻辑在边上
g1.add_edge("order", END)
g1.add_edge("general", END)
app1 = g1.compile()

result = app1.invoke({"messages": [HumanMessage(content="查一下订单")]})
print(f"conditional_edges: {result['messages'][-1].content}")

In [ ]:
# --- 方式 2：Command（动态路由，在节点内部决定） ---

def router_with_command(state: MessagesState) -> Command[Literal["order", "general"]]:
    """路由节点：用 Command 决定下一步"""
    content = state["messages"][-1].content
    if "订单" in content:
        return Command(goto="order")
    return Command(goto="general")


g2 = StateGraph(MessagesState)
g2.add_node("router_node", router_with_command)
g2.add_node("order", handle_order)
g2.add_node("general", handle_general)
g2.add_edge(START, "router_node")
# 不需要 add_conditional_edges，Command 自带路由
g2.add_edge("order", END)
g2.add_edge("general", END)
app2 = g2.compile()

result = app2.invoke({"messages": [HumanMessage(content="查一下订单")]})
print(f"Command: {result['messages'][-1].content}")

### 两种方式对比

| | `conditional_edges` | `Command` |
|--|--------------------|-----------|
| 路由逻辑位置 | 边上（独立的路由函数） | 节点内部 |
| 灵活性 | 只能路由，不能同时改 State | 可以同时路由 + 更新 State |
| 适用场景 | 简单的分支 | 复杂的动态决策（如 Supervisor） |
| 代码风格 | 路由和处理分离 | 路由和处理合一 |

`Command` 更强大，`conditional_edges` 更简单。简单分支用后者，Supervisor 之类的复杂场景用前者。

### Command 的 update 参数

路由的同时更新 State，在一个操作中完成：

In [ ]:
# Command 同时路由和更新 State

def supervisor_with_update(state: MessagesState) -> Command[Literal["worker", "__end__"]]:
    """Supervisor 在路由的同时添加指令消息"""
    return Command(
        goto="worker",
        update={
            "messages": [AIMessage(content="[主管指令] 请重点关注用户情绪，给出安抚性回复。")]
        }
    )


def worker(state: MessagesState) -> dict:
    """工作节点：能看到 Supervisor 附加的指令"""
    # 打印所有消息，验证 Supervisor 的 update 已生效
    for msg in state["messages"]:
        print(f"  {msg.__class__.__name__}: {msg.content[:80]}")
    return {"messages": [AIMessage(content="收到，已处理")]}


g3 = StateGraph(MessagesState)
g3.add_node("supervisor", supervisor_with_update)
g3.add_node("worker", worker)
g3.add_edge(START, "supervisor")
g3.add_edge("worker", END)
app3 = g3.compile()

print("Worker 收到的消息：")
app3.invoke({"messages": [HumanMessage(content="我很生气！")]})
print("\n可以看到 worker 收到了 Supervisor 通过 update 附加的指令消息。")

## 5. Send 并行分发

前面的例子都是**串行**：一次只执行一个 Agent。但有时候需要**并行**：同一个任务交给多个 Agent 同时处理，再汇总结果。

`Send` 可以向同一个节点发送多份不同的输入，实现扇出（fan-out）。

### 示例：多风格写作

场景：给定一个主题，同时让 3 个"写手"用不同风格写一段话（正式、口语、幽默），最后汇总。

In [ ]:
from langgraph.types import Send
from typing import TypedDict, Annotated
import operator

In [ ]:
# 自定义 State：topic 是输入，drafts 用 reducer 收集多个写手的结果

class FanOutState(TypedDict):
    topic: str
    drafts: Annotated[list[str], operator.add]  # reducer: 合并列表


class WriterInput(TypedDict):
    topic: str
    style: str
    drafts: Annotated[list[str], operator.add]

In [ ]:
# 写手节点：根据 style 生成不同风格的文本

def writer(state: WriterInput) -> dict:
    response = llm.invoke([
        SystemMessage(content=f"用{state['style']}的风格，写一句关于以下主题的话（只写一句话）。"),
        HumanMessage(content=state["topic"])
    ])
    return {"drafts": [f"【{state['style']}】{response.content}"]}

In [ ]:
# 路由函数：用 Send 并行分发给同一个 writer 节点，但带不同参数

def fan_out_to_writers(state: FanOutState) -> list[Send]:
    styles = ["正式学术", "轻松口语", "幽默搞笑"]
    return [
        Send("writer", {"topic": state["topic"], "style": s, "drafts": []})
        for s in styles
    ]

In [ ]:
# 汇总节点

def summarize(state: FanOutState) -> dict:
    print("收到的所有稿件：")
    for i, draft in enumerate(state["drafts"], 1):
        print(f"  {i}. {draft}")
    return {}

In [ ]:
# 构建图

fan_builder = StateGraph(FanOutState)
fan_builder.add_node("writer", writer)
fan_builder.add_node("summarize", summarize)

fan_builder.add_conditional_edges(START, fan_out_to_writers)  # Send 扇出
fan_builder.add_edge("writer", "summarize")  # 所有 writer 完成后 → 汇总
fan_builder.add_edge("summarize", END)

fan_out_app = fan_builder.compile()
from IPython.display import Image
Image(fan_out_app.get_graph().draw_mermaid_png())

In [ ]:
# 运行
result = fan_out_app.invoke({"topic": "人工智能的未来", "drafts": []})

print("\n--- 最终 drafts ---")
for draft in result["drafts"]:
    print(draft)

### 刚才发生了什么？

1. `fan_out_to_writers` 返回了 3 个 `Send` 对象，每个指向 `writer` 节点但带不同的 `style` 参数
2. LangGraph **并行**执行 3 个 `writer` 实例（概念上并行，实际可能受 LLM API 限制）
3. 每个 writer 返回 `{"drafts": ["..."]}`，因为 `drafts` 有 `operator.add` reducer，三个结果被**合并**成一个列表
4. 汇总节点收到合并后的完整列表

`Send` 的关键点：
- `Send(node_name, input_data)`：向指定节点发送一份输入
- 多个 `Send` 可以指向**同一个节点**（同一个函数执行多次，不同输入）
- 所有 `Send` 的结果通过 reducer 合并回父 State

## 6. 综合实战：Supervisor + 子图

把前面学的组合起来：用 `create_react_agent` 构建带工具的 Agent 子图，由 Supervisor 调度。

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

In [ ]:
# --- 定义工具 ---

@tool
def search_knowledge(query: str) -> str:
    """搜索知识库获取信息"""
    kb = {
        "退货政策": "7 天无理由退货，需保持商品完好。运费由买家承担，特殊商品除外。",
        "会员等级": "分为普通、银卡、金卡、钻石四级。消费满 1000 升银卡，满 5000 升金卡，满 20000 升钻石。",
        "配送范围": "全国配送，偏远地区加收 10 元运费。港澳台暂不支持。",
    }
    for key, value in kb.items():
        if key in query or any(k in query for k in key):
            return value
    return "未找到相关信息，建议联系人工客服。"


@tool
def lookup_order(order_id: str) -> str:
    """查询订单详情"""
    orders = {
        "ORD100": "商品：蓝牙耳机 | 状态：配送中 | 预计到达：明天",
        "ORD200": "商品：机械键盘 | 状态：已签收 | 签收时间：昨天",
        "ORD300": "商品：显示器 | 状态：待发货 | 预计发货：3天后",
    }
    return orders.get(order_id, f"未找到订单 {order_id}")


@tool
def diagnose_issue(symptom: str) -> str:
    """诊断技术问题"""
    diagnoses = {
        "登录失败": "常见原因：1. 密码错误 2. 账号被冻结 3. 验证码过期。建议先重置密码。",
        "页面空白": "常见原因：1. 浏览器缓存 2. 网络问题 3. 服务器维护。建议清除缓存后重试。",
        "支付失败": "常见原因：1. 余额不足 2. 银行卡限额 3. 支付系统维护中。建议更换支付方式。",
    }
    for key, value in diagnoses.items():
        if key in symptom or any(k in symptom for k in key):
            return value
    return "建议提供更多细节，或联系技术支持热线 400-XXX-XXXX。"

In [ ]:
# --- 用 create_react_agent 创建两个 Agent 子图 ---

# 客服 Agent：处理订单和政策咨询
service_agent = create_react_agent(
    llm,
    [search_knowledge, lookup_order],
    prompt="你是客服 Agent。使用工具回答用户关于订单和政策的问题。简洁回复。",
)

# 技术 Agent：处理技术问题
tech_support_agent = create_react_agent(
    llm,
    [diagnose_issue],
    prompt="你是技术支持 Agent。使用诊断工具帮用户排查技术问题。简洁回复。",
)

In [ ]:
# --- Supervisor 调度 ---

def smart_supervisor(state: MessagesState) -> Command[Literal["service", "tech", "__end__"]]:
    """智能主管：用 LLM 判断路由"""
    response = llm.invoke([
        SystemMessage(content=(
            "你是客服系统的调度主管。根据用户消息和对话历史，决定下一步：\n"
            "- service：订单查询、退货政策、会员等级、配送问题\n"
            "- tech：登录问题、页面故障、支付异常等技术问题\n"
            "- FINISH：问题已被回答，或者是简单问候\n\n"
            "只回复一个词：service 或 tech 或 FINISH"
        )),
        *state["messages"]
    ])
    
    decision = response.content.strip().lower()
    if "service" in decision:
        goto = "service"
    elif "tech" in decision:
        goto = "tech"
    else:
        goto = "__end__"
    
    print(f"  [Supervisor] → {goto}")
    return Command(goto=goto)

In [ ]:
# --- 组装完整的 Supervisor 图 ---

full_builder = StateGraph(MessagesState)
full_builder.add_node("supervisor", smart_supervisor)
full_builder.add_node("service", service_agent)   # create_react_agent 返回的编译图
full_builder.add_node("tech", tech_support_agent)

full_builder.add_edge(START, "supervisor")
full_builder.add_edge("service", "supervisor")
full_builder.add_edge("tech", "supervisor")

full_app = full_builder.compile()
from IPython.display import Image
Image(full_app.get_graph().draw_mermaid_png())

In [ ]:
# 测试：订单查询
result = full_app.invoke({
    "messages": [HumanMessage(content="帮我看看订单 ORD100 的状态")]
})

print("\n--- 对话历史 ---")
for msg in result["messages"]:
    role = "用户" if isinstance(msg, HumanMessage) else "AI"
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"[工具调用] {tc['name']}({tc['args']})")
    elif msg.content:
        print(f"[{role}] {msg.content[:150]}")

In [ ]:
# 测试：技术问题
result = full_app.invoke({
    "messages": [HumanMessage(content="登录一直失败，输了好几次密码都不行")]
})

print("\n--- 最终回复 ---")
# 找最后一条有实质内容的 AI 回复
for msg in reversed(result["messages"]):
    if isinstance(msg, AIMessage) and msg.content and not msg.tool_calls:
        print(msg.content)
        break

### 这个例子的架构

```
父图（Supervisor 调度）
├── supervisor 节点：LLM 决策路由
├── service 节点：create_react_agent 子图（含 search_knowledge + lookup_order 工具）
└── tech 节点：create_react_agent 子图（含 diagnose_issue 工具）
```

子图内部有自己的 ReAct 循环（模型 → 工具 → 模型 → ...），对父图透明。父图只负责调度，不关心子图内部怎么工作。

## 7. 架构模式对比

| 模式 | 协调方式 | 适用场景 | 复杂度 |
|------|---------|---------|--------|
| 顺序子图 | 固定流程 A → B → C | 流水线任务（先收集再写作） | 低 |
| Supervisor | 中央调度，LLM 决策路由 | 客服分流、任务分配 | 中 |
| 并行分发 (Send) | 扇出到多个节点，结果合并 | 同一任务多角度处理 | 中 |
| 对等协作 | Agent 之间互相 `Command(goto=...)` | 复杂多步协商 | 高 |

选择建议：
- **能串行就串行**：最简单，最可预测
- **需要动态分流**：用 Supervisor
- **需要多角度并行**：用 Send
- **Agent 需要自主决定找谁**：对等协作（复杂度最高，慎用）

## 常见问题

**Q1: 子图的 State 和父图不一样怎么办？**

父子图之间只有**同名 key** 会自动映射。如果子图有额外的 key，不会影响父图。如果父图有子图没有的 key，子图也不会收到。确保需要传递的数据用相同的 key 名。

**Q2: Supervisor 的 LLM 调用会不会太多？**

每次循环（Agent 完成 → 回到 Supervisor）都会调用一次 LLM 做决策。如果 Agent 很多或循环很多次，确实会增加开销。可以通过以下方式优化：
- 让 Supervisor 的 prompt 更精确，减少犹豫
- 用更快/便宜的模型做路由判断
- 设置最大循环次数（`recursion_limit`）

**Q3: Send 是真正的并行吗？**

LangGraph 会并行调度 Send 的节点执行。但如果节点内部调用 LLM API，实际并行度取决于 API 的并发限制。对于 CPU 密集型操作，Send 的并行优势更明显。

**Q4: Command 的类型标注 `Command[Literal[...]]` 是必须的吗？**

是的。LangGraph 需要通过返回类型标注知道这个节点可能路由到哪些节点，才能正确构建图的结构。如果不标注，图的构建会缺少必要的边信息。

## 总结

本章学习了多 Agent 协作的核心技术：

- **子图封装**：把 Agent 编译为独立的图，作为父图的节点使用，父子图通过同名 State key 自动传递数据
- **Supervisor 模式**：用一个中央节点（LLM 驱动）动态路由请求到对应 Agent，是最常用的多 Agent 架构
- **Command(goto=...)**：在节点内部决定路由，比 `conditional_edges` 更灵活，支持同时更新 State
- **Send 并行分发**：向同一个节点发送多份不同输入，实现扇出，结果通过 reducer 合并

## 下一章预告

**第七章：持久化与检查点** —— 目前图的状态都在内存中，程序一关就没了。下一章学习如何持久化保存状态，实现多轮对话、断点续跑和时间旅行。